# 08. Mini-Capstone: an Automated Processing Workflow
**Engineering Data Processing with Python** | one-day course | about 45 minutes

> **TRAINER / SOLUTION VERSION.** Every blank is filled in and the notebook has been run.


In [1]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## The brief

> **From:** Operations Manager, Kilbrack Engineering
> **Subject:** Galway September numbers
>
> Galway uses a different maintenance system from Cork. Their September 2026 export is attached. By Friday I need:
> a clean dataset, a list of anything Galway needs to fix, work orders by type, average time to get broken machines
> running again by asset type, and the five assets with the most breakdowns. Plus a short note on anything I should
> be careful about. I'll want this every month, so please don't do it by hand.

You will process Galway's file with the **same script** as Module 07, changing only the settings.

| Step | What you do | Time |
|---|---|---|
| 1 | Look at the raw file and spot what is different | 10 min |
| 2 | Write Galway's settings | 10 min |
| 3 | Run it, read the error, fix the settings, run again | 5 min |
| 4 | Work out the three figures and save them to Excel | 10 min |
| 5 | Write the note for the manager | 10 min |

In [2]:
import shutil, json, sys
from pathlib import Path
shutil.copy("data/checkpoints/pipeline.py", "pipeline.py")
shutil.copy("data/checkpoints/config_cmms.json", "config_cmms.json")
with open("config_cmms.json", encoding="utf-8") as f:
    cmms_config = json.load(f)
print("Script and Cork settings ready.")

Script and Cork settings ready.


## 1. Look at the raw file first

Open it as plain text before pandas interprets anything. Run and read.

In [3]:
RAW = "data/capstone/wo_galway_2026_09.csv"
with open(RAW, encoding="utf-8") as f:
    for line in f.readlines()[:4]:
        print(line.rstrip())

WorkOrder;AssetTag;Category;Prio;Opened;Closed;Hours;Cost EUR;Tech;Details;State;Modified
GWY-0027;CMP-003;Breakdown;3;06/09/2026 06:45;2026-09-06T21:56:39;3,5;637,66;Declan Murphy;Oil leak at separator;Closed;2026-09-06T23:56:39
GWY-0124;CNC-001;Planned Maint.;4;2026-09-25T12:00:00;;6;211,08;D. Murphy;Grease bearings, check alignment;Open;2026-09-25T14:00:00
GWY-0075;HYP-008;Planned Maint.;3;2026-09-15T08:45:00;2026-09-17T12:39:03;4,5;181,78;Murphy, Declan;Grease bearings, check alignment;Closed;2026-09-17T14:39:03


Spot the differences from the Cork file. Discuss with your neighbour before opening the answer.

1. What separates the columns?
2. How is half an hour written in the `Hours` column?
3. Are the column names the same as Cork's?

<details><summary><b>Show the answer</b></summary>

1. A semicolon `;` instead of a comma.
2. `3,5`: a **comma** is the decimal mark. Cost `1.250,00` uses a dot for thousands.
3. No: `WorkOrder`, `AssetTag`, `Opened`... instead of `wo_id`, `asset_id`, `date_raised`.
Also, dates look like `2026-09-25T12:00:00`, with a few `06/09/2026 06:45`.
</details>

In [4]:
galway_raw = pd.read_csv(RAW, sep=";")
print(galway_raw.shape)
galway_raw["Category"].value_counts()

(164, 12)


Category
Planned Maint.    38
PPM               30
PM                26
Reactive          20
CM                17
Breakdown         15
INSP               9
Inspection         9
Name: count, dtype: int64

## 2. Galway's settings

We copy Cork's settings and change only what is different.

### Your turn 8.1
Fill in the separator (`;`), the decimal mark (`,`) and the three missing column names.
Our names are: `wo_id`, `asset_id`, `wo_type`, `priority`, `date_raised`, `date_closed`, `labour_hours`,
`parts_cost_eur`, `technician`, `description`, `status`, `last_modified`.

In [5]:
galway = json.loads(json.dumps(cmms_config))       # full copy of the Cork settings
galway["run_name"] = "galway_2026_09"
galway["source"]["path"] = RAW
galway["source"]["sep"] = ";"
galway["source"]["decimal"] = ","
galway["source"]["date_formats"] = ["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M"]
galway["source"]["modified_formats"] = ["%Y-%m-%dT%H:%M:%S"]
galway["source"]["column_map"] = {
    "WorkOrder": "wo_id",
    "AssetTag": "asset_id",
    "Category": "wo_type",
    "Prio": "priority",
    "Opened": "date_raised",
    "Closed": "date_closed",
    "Hours": "labour_hours",
    "Cost EUR": "parts_cost_eur",
    "Tech": "technician",
    "Details": "description",
    "State": "status",
    "Modified": "last_modified",
}

def save_galway():
    with open("config_galway.json", "w", encoding="utf-8") as f:
        json.dump(galway, f, indent=2, ensure_ascii=False)
save_galway()
print("Saved config_galway.json")

Saved config_galway.json


## 3. Run it, and let it fail properly

In [6]:
!{sys.executable} pipeline.py config_galway.json

2026-10-06 14:38:28,516 INFO    Run 'galway_2026_09' started
2026-10-06 14:38:28,519 INFO    Loaded 164 rows from data/capstone/wo_galway_2026_09.csv
Traceback (most recent call last):
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 211, in <module>
    run_pipeline(config)
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 196, in run_pipeline
    df = clean_work_orders(df, cfg)
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 85, in clean_work_orders
    raise ValueError(f"Unmapped work order types {unmapped}. Add them to type_map in the config.")
ValueError: Unmapped work order types ['PLANNED MAINT.', 'PPM', 'REACTIVE']. Add them to type_map in the config.


It stops with **Unmapped work order types** and lists the words it does not know: `PPM`, `PLANNED MAINT.`,
`REACTIVE`. That is the script doing its job: it refuses to guess what Galway means by "Reactive". You decide.

### Your turn 8.2
`PPM` and `PLANNED MAINT.` are planned maintenance (`PM`). `REACTIVE` is a breakdown (`CM`).

In [7]:
galway["type_map"]["PPM"] = "PM"
galway["type_map"]["PLANNED MAINT."] = "PM"
galway["type_map"]["REACTIVE"] = "CM"
save_galway()

!{sys.executable} pipeline.py config_galway.json

2026-10-06 14:38:29,304 INFO    Run 'galway_2026_09' started
2026-10-06 14:38:29,307 INFO    Loaded 164 rows from data/capstone/wo_galway_2026_09.csv


2026-10-06 14:38:29,318 INFO    Dedup: 6 exact duplicates, 8 older versions removed, 150 work orders


2026-10-06 14:38:29,494 WARNING Rule closed_before_raised: 1 rows
2026-10-06 14:38:29,495 WARNING Rule negative_hours: 1 rows
2026-10-06 14:38:29,495 WARNING Rule implausible_hours: 1 rows
2026-10-06 14:38:29,496 WARNING Rule unknown_asset: 2 rows
2026-10-06 14:38:29,496 WARNING Rule implausible_cost: 1 rows
2026-10-06 14:38:29,496 INFO    Rule missing_hours: 1 rows
2026-10-06 14:38:29,510 INFO    Outputs written to outputs/galway_2026_09
2026-10-06 14:38:29,510 INFO    Finished: {'rows_out': 150, 'errors': 6, 'output_dir': 'outputs/galway_2026_09'}


In [8]:
run_dir = Path("outputs/galway_2026_09")
check((run_dir / "dq_report.csv").exists(), "Galway processed by changing settings only.",
      "No outputs yet. Read the last line of the error above: it says which setting to fix.")

PASS: Galway processed by changing settings only.


Read the log above: how many duplicates were removed, and which rules fired? Now look at the rows with errors.
These go back to Galway to fix:

In [9]:
issues = pd.read_csv(run_dir / "dq_issues.csv")
issues[issues["dq_error"]][["wo_id", "asset_id", "date_raised", "date_closed", "labour_hours", "parts_cost_eur", "dq_issues"]]

,wo_id,asset_id,date_raised,date_closed,labour_hours,parts_cost_eur,dq_issues
0,GWY-0006,CMP-007,2026-09-01 16:20:00,2026-09-02 12:25:35,300.0,1081.58,implausible_hours
1,GWY-0018,CMP-003,2026-09-03 17:50:00,2026-09-04 16:18:43,-3.0,67.06,negative_hours
2,GWY-0023,CNC-013,2026-09-04 13:00:00,2026-09-06 16:56:17,3.5,99999.00,implausible_cost
3,GWY-0031,CHL-099,2026-09-06 11:30:00,2026-09-08 21:28:34,4.0,17.25,unknown_asset
4,GWY-0042,PMP-099,2026-09-08 16:00:00,2026-09-10 16:36:31,5.5,278.77,unknown_asset
5,GWY-0049,CNC-014,2026-09-10 12:50:00,2026-08-30 10:00:00,4.0,228.25,closed_before_raised


Is 300 hours a typo for 3.00, or 300 minutes? You cannot know. That is why it goes back to Galway rather than being
"fixed" by you.

## 4. The three figures

Only rows **without errors** go into the figures. Example: work orders by type.

In [10]:
ready = pd.read_csv(run_dir / "work_orders_kpi_ready.csv", parse_dates=["date_raised", "date_closed"])
by_type = ready["wo_type"].value_counts()
by_type

wo_type
PM      79
CM      48
INSP    17
Name: count, dtype: int64

**Time to get a machine running again** = closed time minus raised time, for closed breakdown (`CM`) jobs.

### Your turn 8.3
Pick out breakdown jobs: the type code is `CM`.

In [11]:
breakdowns = ready[(ready["wo_type"] == "CM") & ready["date_closed"].notna()].copy()
breakdowns["hours_to_restore"] = (breakdowns["date_closed"] - breakdowns["date_raised"]).dt.total_seconds() / 3600

restore = breakdowns.groupby("asset_type")["hours_to_restore"].agg(["count", "mean", "median"]).round(1)
restore

,count,mean,median
asset_type,,,
Air Compressor,8,32.4,31.9
CNC Mill,20,21.9,20.8
Conveyor,3,27.7,17.4
Hydraulic Press,2,11.1,11.1
Pump,6,22.8,21.7


Why show the median as well as the mean? One very long job can drag the average up. If the two are far apart,
a few jobs are unusual.

### Your turn 8.4
The five assets with the most breakdowns. `.head(n)` shows the first n rows.

In [12]:
top_assets = breakdowns["asset_id"].value_counts().head(5)
top_assets

asset_id
CNC-012    4
CNC-007    4
CMP-008    4
CMP-003    4
CNC-004    4
Name: count, dtype: int64

In [13]:
check(len(top_assets) == 5, "Top five found.", "Expected 5 assets.")

PASS: Top five found.


Save all three to one Excel file for the manager:

In [14]:
with pd.ExcelWriter(run_dir / "galway_kpis.xlsx") as xl:
    by_type.to_excel(xl, sheet_name="work orders by type")
    restore.to_excel(xl, sheet_name="hours to restore")
    top_assets.to_excel(xl, sheet_name="top 5 breakdown assets")
print("Saved", run_dir / "galway_kpis.xlsx", "(download it from the Files panel)")

Saved outputs/galway_2026_09/galway_kpis.xlsx (download it from the Files panel)


## 5. The note for the manager

Plain English, five to eight lines. Use the log and the tables above to fill in the numbers. Double-click the cell
below to edit it.

**Optional, with Gemini:** paste your note and ask *"You are an operations manager. What in this note is unclear,
and what would you want to know that it doesn't say?"* Improve it, but keep it yours.

**Galway work orders, September 2026** (example)

* Rows received: 164. Duplicates removed: 14 (6 exact copies, 8 older versions of jobs later closed). Work orders: 150.
* Left out of the figures because of errors: 6. Two are on asset tags not in the register (CHL-099, PMP-099),
  one has 300 labour hours, one has -3 hours, one has a parts cost of €99,999 (looks like a placeholder) and one was
  closed before it was opened. These are listed in `dq_issues.csv` for Galway to fix.
* I assumed: "Reactive" means breakdown; "PPM" and "Planned Maint." mean planned maintenance.
* Be careful: "hours to restore" runs from when the job was raised to when it was closed, so it includes waiting for
  parts and people, not just repair time. Breakdowns still open at month end are not included yet.

## What you did today

You went from "the file loads" to a workflow that:

* reads differently formatted exports by changing **settings**, not code,
* tidies text, spellings, numbers, units and dates, and counts what it could not convert,
* removes duplicates by a rule you can explain, and sends conflicts to a person,
* combines sources without double counting, and keeps track of where every row came from,
* checks quality with clear rules, and stops when something is fundamentally wrong,
* produces the same outputs, the same way, every month.

And you used AI as a helper whose work you checked, not as an oracle.